# bio-kg-triage: an ontology-grounded biomedical KG, validated

Every edge in this knowledge graph is typed with the real **Biolink Model** vocabulary and passes a **closed-world vocabulary gate** before it enters the graph, the same gate benchmarked on schema.org and IES4 in [onto-correctness-bench](https://gov.tesseract.academy/research/ontology-correctness-benchmark). Data is live from the **Open Targets Platform**.

**Press Runtime then Run all.** It builds a gene-disease KG from real Open Targets associations, then shows the grounded KG has 0 vocabulary violations while an ungrounded variant (one fabricated Biolink predicate) slips past SHACL but is caught by the gate.

Code: https://github.com/fabio-rovai/open-ontologies/tree/main/case-studies/bio-kg-triage


In [ ]:
!pip -q install rdflib pyshacl requests


In [ ]:
VOCAB_URL = "https://raw.githubusercontent.com/fabio-rovai/open-ontologies/main/case-studies/bio-kg-triage/data/biolink_vocab.json"
import json, requests, rdflib
from rdflib import Graph, RDF, RDFS, URIRef, Literal
from rdflib.namespace import SH
from pyshacl import validate

BL = "https://w3id.org/biolink/vocab/"
V = requests.get(VOCAB_URL, timeout=60).json()
DECLARED = set(V["declared"]); POLICED = V["policed"]
print("Biolink declared terms:", len(DECLARED))

TARGETS = {"ENSG00000146648":"EGFR","ENSG00000141510":"TP53","ENSG00000133703":"KRAS",
 "ENSG00000012048":"BRCA1","ENSG00000171862":"PTEN","ENSG00000157764":"BRAF",
 "ENSG00000171094":"ALK","ENSG00000136997":"MYC"}
OT = "https://api.platform.opentargets.org/api/v4/graphql"
Q = '{ target(ensemblId:"ID"){ approvedSymbol associatedDiseases(page:{index:0,size:5}){ rows{ disease{ id name } score } } } }'
rows = []
for ens, sym in TARGETS.items():
    r = requests.post(OT, json={"query": Q.replace("ID", ens)}, timeout=60).json()
    t = (r.get("data") or {}).get("target") or {}
    for a in (t.get("associatedDiseases") or {}).get("rows", []):
        rows.append({"ensembl":ens, "symbol":t.get("approvedSymbol",sym),
                     "disease_id":a["disease"]["id"], "disease_name":a["disease"]["name"], "score":round(float(a["score"]),4)})
print("fetched", len(rows), "real gene-disease associations from Open Targets")

def in_pol(i): return any(str(i).startswith(p) for p in POLICED)
def gate(g):
    f = set()
    for s,p,o in g:
        if in_pol(p) and str(p) not in DECLARED: f.add(str(p))
        if p==RDF.type and isinstance(o,URIRef) and in_pol(o) and str(o) not in DECLARED: f.add(str(o))
    return sorted(f)
def dis_iri(d):
    return "http://purl.obolibrary.org/obo/"+d if d.startswith(("MONDO_","HP_","GO_")) else "https://identifiers.org/"+d.replace("_",":",1).lower()
def build(pred):
    g = Graph()
    for r in rows:
        ge = URIRef("https://identifiers.org/ensembl:"+r["ensembl"]); di = URIRef(dis_iri(r["disease_id"]))
        g.add((ge,RDF.type,URIRef(BL+"Gene"))); g.add((ge,RDFS.label,Literal(r["symbol"])))
        g.add((di,RDF.type,URIRef(BL+"Disease"))); g.add((di,RDFS.label,Literal(r["disease_name"])))
        g.add((ge,URIRef(pred),di))
    return g
def shapes():
    s = Graph(); sh = URIRef("https://ex/shape")
    s.add((sh,RDF.type,SH.NodeShape)); s.add((sh,SH.targetClass,URIRef(BL+"Gene")))
    b = URIRef("https://ex/shape/l"); s.add((sh,SH.property,b)); s.add((b,SH.path,RDFS.label)); s.add((b,SH.minCount,Literal(1)))
    return s
def evalg(name, g):
    c,_,_ = validate(g, shacl_graph=shapes(), inference="none")
    fl = gate(g)
    print(name, "triples=", len(g), " SHACL conforms=", str(c).lower(), " closed-world violations=", len(fl), fl)

evalg("grounded  ", build(BL+"gene_associated_with_condition"))   # real Biolink predicate
evalg("ungrounded", build(BL+"associated_with_disease"))          # fabricated, not declared


In [ ]:
print("Top gene-disease hypotheses by Open Targets score (each a validated Biolink triple):\n")
for i, r in enumerate(sorted(rows, key=lambda x: x["score"], reverse=True)[:10], 1):
    print(str(i).rjust(2) + ". " + r["symbol"].ljust(6) + " -> " + r["disease_name"].ljust(45) + " " + str(r["score"]))


---
The grounded KG has **0 SHACL and 0 closed-world violations**; the ungrounded variant passes SHACL but the gate catches the fabricated predicate. Read the write-up: https://gov.tesseract.academy/research/ontology-grounded-biomedical-kg

Built by Tesseract Academy.
